# 🧠 Sesiunea 22 · Embeddings și Transformers

**Întrebarea de azi:** cum poate un calculator să înțeleagă că *„livrarea a fost rapidă”* și *„coletul a ajuns repede”* spun același lucru?

Drumul:
1. De ce numărarea cuvintelor (TF-IDF) nu ajunge
2. **Embeddings**: fiecare text devine un punct pe o „hartă a sensului”
3. Căutare după sens în **recenzii reale** în română
4. **Embeddings contextuale**: același cuvânt, alt sens
5. **Transformer**: cum folosește modelul contextul
6. 🧪 Do it yourself

## Pregătire (rulează o dată)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

O funcție ajutătoare: cât de **asemănători** sunt doi vectori. Aproape de **1** = foarte asemănători, aproape de **0** = fără legătură.

In [ ]:
def similaritate(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 1 · Problema cu numărarea cuvintelor 🔢
*(după slide-ul „Ce știm deja: TF-IDF”)*

În sesiunea de NLP am transformat textul în numere **numărând cuvintele** (TF-IDF). Hai să vedem ce crede TF-IDF despre două fraze cu același sens.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

fraze = ["livrarea a fost rapida", "coletul a ajuns repede"]
tfidf = TfidfVectorizer().fit_transform(fraze).toarray()

In [ ]:
similaritate(tfidf[0], tfidf[1])

**🤔 Întrebare** · Cele două fraze spun același lucru. De ce crede TF-IDF că nu au **nicio** legătură (0)?

> ✍️ *Răspunsul tău:* ....

## 2 · Embeddings: textul devine un punct pe o hartă 🗺️
*(după slide-urile „Embeddings”)*

Un **embedding** = o listă de numere care descrie **sensul** unui text. Texte cu sens apropiat primesc numere apropiate.

Folosim un model gata antrenat de alții, care știe și limba română. Prima rulare îl descarcă (~0,5 GB).

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Transformăm un cuvânt în embedding:

In [ ]:
v = model.encode("pisica")
v.shape

In [ ]:
v[:10]

**🤔 Întrebare** · Câte numere descriu cuvântul? Ce înseamnă al 3-lea număr?

> ✍️ *Răspunsul tău:* ....

O funcție mică: cât de aproape sunt două texte, ca **sens**.

In [ ]:
def cat_de_aproape(text1, text2):
    return round(similaritate(model.encode(text1), model.encode(text2)), 2)

In [ ]:
cat_de_aproape("pisica", "caine")

In [ ]:
cat_de_aproape("pisica", "masina")

**Exercițiul 2.1 · Joacă-te cu perechi**

Încearcă 3 perechi ale tale: două cuvinte apropiate ca sens (ex: „medic” și „doctor”), două fără legătură, două opuse (ex: „cald” și „rece”). Ce observi la cele opuse?

> ✍️ *Răspunsul tău:* ....

In [ ]:
cat_de_aproape("medic", "doctor")      # ✏️ schimbă perechile

### Harta sensului
Folosim cuvinte uzuale în engleză și două axe simple bazate pe apropierea de `animals`, `food` și `transport`.

In [ ]:
cuvinte = ["cat", "dog", "horse", "cow", "bread", "cheese", "apple", "soup", "car", "bus", "train", "bicycle"]
categorii = ["animals"] * 4 + ["food"] * 4 + ["transport"] * 4
vectori = model.encode(cuvinte, normalize_embeddings=True)

In [ ]:
repere = model.encode(["animals", "food", "transport"], normalize_embeddings=True)
puncte = np.c_[vectori @ repere[0] - vectori @ repere[2], vectori @ repere[1] - vectori @ repere[2]]

In [ ]:
culori = {"animals": "#2E86AB", "food": "#F18F01", "transport": "#6A994E"}
for categorie, culoare in culori.items():
    i = [j for j, c in enumerate(categorii) if c == categorie]; plt.scatter(puncte[i, 0], puncte[i, 1], label=categorie, color=culoare)
for cuvant, (x, y) in zip(cuvinte, puncte): plt.annotate(cuvant, (x, y))
plt.xlabel("transport ← 0 → animals"); plt.ylabel("transport ← 0 → food"); plt.legend(); plt.show()

**🤔 Întrebare** · Nu i-am spus modelului care cuvinte sunt animale, mâncare sau vehicule. Cum s-au grupat totuși?

> ✍️ *Răspunsul tău:* ....

## 3 · Același sens, alte cuvinte 💬
*(după slide-ul „Propoziții, nu doar cuvinte”)*

Embeddings funcționează și pentru **propoziții întregi**. Revenim la frazele de la început:

In [ ]:
cat_de_aproape("livrarea a fost rapida", "coletul a ajuns repede")

In [ ]:
cat_de_aproape("livrarea a fost rapida", "produsul s-a stricat dupa o zi")

**Exercițiul 3.1 · TF-IDF vs embeddings**

Calculează cât de aproape sunt *„telefonul se descarca foarte repede”* și *„bateria tine putin”*. Ce ar fi spus TF-IDF?

> ✍️ *Răspunsul tău:* ....

In [ ]:
# ✏️ calculează cât de aproape sunt cele două fraze

## 4 · Căutare după sens în recenzii reale 🔎
*(după slide-ul „Căutare după sens”)*

Recenzii reale în română. Folosim un eșantion mai mare, ca să găsim mai ușor rezultate relevante.

In [ ]:
URL = "https://raw.githubusercontent.com/dumitrescustefan/Romanian-Transformers/examples/examples/sentiment_analysis/ro/train.csv"
recenzii = pd.read_csv(URL)["text"].dropna().sample(5000, random_state=0).tolist()

Transformăm toate recenziile în embeddings (durează puțin; cu GPU, câteva secunde).

In [ ]:
vectori_recenzii = model.encode(recenzii, normalize_embeddings=True, show_progress_bar=True)
vectori_recenzii.shape

O funcție ajutătoare: caută recenziile cele mai apropiate **ca sens** de o întrebare. Citește doar comentariile.

In [ ]:
def cauta(intrebare, cate=3):
    q = model.encode(intrebare, normalize_embeddings=True)
    scoruri = vectori_recenzii @ q
    for i in np.argsort(scoruri)[::-1][:cate]: print(round(float(scoruri[i]), 2), "|", recenzii[i][:150])

In [ ]:
cauta("bateria tine putin")

**Exercițiul 4.1 · Caută tu**

Încearcă 3 căutări ale tale (ex: „livrare intarziata”, „film plictisitor”, „raport calitate pret excelent”). Conțin rezultatele **exact** cuvintele tale?

> ✍️ *Răspunsul tău:* ....

In [ ]:
cauta("livrare intarziata")       # ✏️ caută ce vrei tu

## 5 · Același cuvânt, alt sens 🏦🪑
*(după slide-ul „Embeddings contextuale”)*

„Banca” poate fi o **instituție** (iei un credit) sau un **scaun în parc**. Dacă dăm modelului doar cuvântul, primește mereu **aceeași** listă de numere, oricare ar fi sensul:

In [ ]:
cat_de_aproape("banca", "banca")

Un model **contextual** citește toată propoziția și dă cuvântului un vector **care depinde de context**. Folosim **XLM-RoBERTa**, un Transformer care știe și limba română (~1 GB la prima rulare).

Celula de mai jos e ajutătoare: definește `vector_cuvant(propozitie, cuvant)`. Nu trebuie s-o înțelegi.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
tok_ctx = AutoTokenizer.from_pretrained("xlm-roberta-base")
model_ctx = AutoModel.from_pretrained("xlm-roberta-base")

def vector_cuvant(propozitie, cuvant):
    intrare = tok_ctx(propozitie, return_tensors="pt")
    tokeni = tok_ctx.convert_ids_to_tokens(intrare["input_ids"][0])
    with torch.no_grad(): stari = model_ctx(**intrare).last_hidden_state[0]
    pozitii = [i for i, token in enumerate(tokeni) if cuvant.lower() in token.lower()]
    return stari[pozitii].mean(0).numpy()

def tabel_similaritati(vectori, etichete):
    vectori = np.asarray(vectori, dtype=float); vectori -= vectori.mean(0)
    vectori /= np.linalg.norm(vectori, axis=1, keepdims=True)
    return pd.DataFrame(np.round(vectori @ vectori.T, 2), index=etichete, columns=etichete)

Patru propoziții: două cu banca-**instituție**, două cu banca-**scaun**.

In [ ]:
propozitii = ["Maine merg la banca sa iau un credit.",
              "Am sunat la banca pentru un card nou.",
              "M-am odihnit pe banca din parc.",
              "Copiii s-au asezat pe banca de langa lac."]

In [ ]:
vectori_banca = [vector_cuvant(p, "banca") for p in propozitii]

In [ ]:
tabel_similaritati(vectori_banca, ["credit", "card", "parc", "lac"])

**🤔 Întrebare** · Uită-te la tabel. Care perechi de „banca” sunt cele mai apropiate? Ce înseamnă asta?

> ✍️ *Răspunsul tău:* ....

**Exercițiul 5.1 · Cuvântul tău cu două sensuri**

Alege un cuvânt cu două sensuri: „lac” (apă / vopsea), „broasca” (animal / încuietoare), „toc” (pantof / ușă), „mouse” (animal / calculator). Scrie 4 propoziții, două pentru fiecare sens, și refă tabelul.

> ✍️ *Răspunsul tău:* ....

In [ ]:
cuvantul_meu = "lac"
propozitiile_mele = ["Am pescuit pe lac.", "Am inotat in lac.",
                     "Am dat cu lac pe usa.", "Lac de unghii rosu."]
vectori_mei = [vector_cuvant(p, cuvantul_meu) for p in propozitiile_mele]
tabel_similaritati(vectori_mei, ["1", "2", "3", "4"])

## 6 · Transformer: cum folosește contextul 🔁
*(după slide-urile „Transformer” și „Atenția”)*

Un Transformer citește **toate cuvintele deodată**, iar fiecare cuvânt „se uită” la celelalte ca să-și înțeleagă sensul (mecanismul de **atenție**).

Îl testăm cu un joc: ascundem un cuvânt (`<mask>`) și modelul ghicește ce lipsește.

In [ ]:
from transformers import pipeline
completeaza = pipeline("fill-mask", model="xlm-roberta-base")

O funcție mică: afișează cele mai probabile 5 cuvinte.

In [ ]:
def ghiceste(propozitie):
    return pd.DataFrame(completeaza(propozitie))[["token_str", "score"]].round(2)

In [ ]:
ghiceste("Am mers la <mask> sa cumpar paine.")

In [ ]:
ghiceste("Am mers la <mask> sa scot bani.")

**🤔 Întrebare** · Am schimbat doar **ultimul** cuvânt („paine” → „bani”). De ce s-a schimbat ghicitul pentru cuvântul din **mijloc**?

> ✍️ *Răspunsul tău:* ....

**Exercițiul 6.1 · Păcălește modelul**

Scrie 2 propoziții cu `<mask>` în care **un singur cuvânt** de departe schimbă complet răspunsul. Exemplu: „Pisica a prins un <mask> in gradina.” vs „Pisica a prins un <mask> in apa.”

> ✍️ *Răspunsul tău:* ....

In [ ]:
ghiceste("Pisica a prins un <mask> in gradina.")      # ✏️ propoziția ta

## Recap 🎓

- **TF-IDF** numără cuvinte: „rapida” și „repede” nu au nicio legătură pentru el.
- **Embedding** = listă de numere care descrie **sensul**; texte apropiate ca sens → numere apropiate.
- **Similaritate** aproape de 1 = sens apropiat. Merge pentru cuvinte, propoziții, recenzii întregi.
- **Căutare după sens**: găsești texte care spun același lucru cu alte cuvinte.
- **Embedding contextual**: același cuvânt primește vectori diferiți după propoziție („banca” din parc vs „banca” cu credite).
- **Transformer**: citește toate cuvintele deodată; prin **atenție**, fiecare cuvânt se uită la celelalte. Stă la baza ChatGPT, Claude, Gemini.

# 🧪 Do it yourself
Alege 2–3 exerciții. Nu este mini-proiect; testează ideile în pași mici.

**Exercițiul 1 · Perechi.** Găsește o pereche apropiată și una fără legătură. Compară scorurile.

In [ ]:
cat_de_aproape("medic", "doctor")
cat_de_aproape("medic", "autobuz")

In [ ]:
# ✏️ Înlocuiește cu perechile tale

**Exercițiul 2 · Căutare.** Schimbă întrebarea și observă rezultatele.

In [ ]:
cauta("camera telefonului este slaba")
# ✏️ Încearcă: "livrare rapida" sau "film plictisitor"

**Exercițiul 3 · Context.** Schimbă ultimul cuvânt și compară predicțiile.

In [ ]:
ghiceste("Am mers la <mask> sa cumpar paine.")
ghiceste("Am mers la <mask> sa scot bani.")

**Exercițiul 4 · Două sensuri.** Înlocuiește propozițiile și compară vectorii.

In [ ]:
propozitii_mele = ["Am pescuit pe lac.", "Am dat cu lac pe usa."]
vectori_mei = [vector_cuvant(p, "lac") for p in propozitii_mele]
tabel_similaritati(vectori_mei, ["apa", "vopsea"])

**Exercițiul 5 · Capcane.** Scrie o propoziție cu negație și una sarcastică.

In [ ]:
capcane_mele = ["nu este rau deloc", "super, s-a stricat imediat"]
# ✏️ Discută ce ar putea greși un model

**Exercițiul 6 · Întrebare.** De ce nu este suficientă căutarea exactă?

In [ ]:
# ✏️ Răspunde în 2–3 rânduri cu un exemplu din căutare.

**Reflecție finală**

> ✍️ Notează un rezultat surprinzător și explică-l în 2–3 rânduri.

**Notițele tale**

> ✍️ *Scrie aici observațiile tale:* ....